# Bronze Layer
Raw ingestion from the landing zone into Delta tables. No transformations — just load the data as-is and add metadata for traceability.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import uuid
import os

repo_root = os.getcwd().split("/01_bronze")[0]

# landing zone in Unity Catalog Volume
landing_base = "/Volumes/dev/electroflow_pipeline/landing_data"

# source files to ingest
source_files = [
    {"name": "customers", "file": "customers.csv", "format": "csv"},
    {"name": "products",  "file": "products.csv",  "format": "csv"},
    {"name": "orders",    "file": "orders.json",    "format": "json"},
    {"name": "payments",  "file": "order_payments.csv", "format": "csv"},
    {"name": "coupons",   "file": "coupons.csv",   "format": "csv"}
]

In [0]:
def load_raw_to_bronze(source_path, target_table, file_format='csv'):
    print(f"Loading {source_path} to {target_table}")

    # read raw data
    if file_format == "csv":
        df = spark.read.format("csv")\
            .option("header","true")\
            .option("inferschema","true")\
            .load(source_path)
    elif file_format == "json":
        df = spark.read.format('json')\
            .option("inferschema","true")\
            .option("multiline", "true")\
            .load(source_path) 

    # add metadata columns
    df_enriched = df.select("*", F.col("_metadata.file_path").alias("_source_file_path")) \
                    .withColumn("_ingestion_timestamp", F.current_timestamp()) \
                    .withColumn("_ingestion_job_id", F.lit(str(uuid.uuid4())))

    # write as managed table in Unity Catalog
    target_table_name = f"dev.electroflow_pipeline.{target_table}"
    df_enriched.write.format("delta")\
        .mode("overwrite")\
        .saveAsTable(target_table_name)

    print(f"Done: {target_table}")
    return df_enriched

# run for all source files
for config in source_files:
    source_path = f"{landing_base}/{config['file']}"
    target_table = f"bronze_{config['name']}"
    load_raw_to_bronze(source_path, target_table, file_format=config['format'])